In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.2996, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda', index=1)

### Load Trained Weights

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
from train_eval_func import eval_loop

In [28]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

### Load Trained Weights for Model with 7 Layers

In [29]:
import os
from merge_helper import merge_bert_layers, LayerMergeTracker, apply_layer_tracking_to_model
from train_eval_func import train, get_primary_metric, set_lr_scheduler, get_recovery_epoch
from torch.optim import AdamW

In [30]:
model_7 = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
load_graft_path = '../similar_layer_merge/weights/'
graft_state = torch.load(load_graft_path + f'layer-7-{task_name}.pt', map_location=device)

In [32]:
model_7 = apply_layer_tracking_to_model(model_7, graft_state['layer_track'])

In [33]:
len(model_7.bert.encoder.layer)

7

In [34]:
model_7.to(device)
device

device(type='cuda', index=1)

In [35]:
model_7.load_state_dict(graft_state['model_state_dict'])

<All keys matched successfully>

In [36]:
eval_loop(model_7, val_dataloader, task_name, device)[0]

{'accuracy': 0.8700980392156863, 'f1': 0.9084628670120898}

# 5. Helper Functions

In [37]:
import torch.nn as nn
from tqdm import tqdm
from typing import Optional, Tuple, List

In [38]:
def register_importance_masks(model, device, register_heads=True, register_ffn=True):
    """
    Register importance masks for heads and/or FFN.
    
    Flexible registration - can register only heads, only FFN, or both.
    
    Args:
        model: BERT model
        device: torch device
        register_heads: Whether to register head masks (default: True)
        register_ffn: Whether to register FFN masks (default: True)
    
    Examples:
        # Register both (most common)
        register_importance_masks(model, device)
        
        # Register only heads
        register_importance_masks(model, device, register_heads=True, register_ffn=False)
        
        # Register only FFN
        register_importance_masks(model, device, register_heads=False, register_ffn=True)
    """
    if not register_heads and not register_ffn:
        raise ValueError("Must register at least one of: heads or FFN")
    
    for layer_idx, layer in enumerate(model.bert.encoder.layer):
        if register_heads:
            n_heads = layer.attention.self.num_attention_heads
            layer.head_mask_param = nn.Parameter(
                torch.ones(n_heads, dtype=torch.float32, device=device),
                requires_grad=True
            )
        
        if register_ffn:
            ffn_dim = layer.intermediate.dense.out_features
            layer.int_mask_param = nn.Parameter(
                torch.ones(ffn_dim, dtype=torch.float32, device=device),
                requires_grad=True
            )
        
        # Print registration info
        parts = []
        if register_heads:
            parts.append(f"{layer.attention.self.num_attention_heads} heads")
        if register_ffn:
            parts.append(f"{layer.intermediate.dense.out_features} neurons")
        
        print(f"  Layer {layer_idx}: Registered masks ({', '.join(parts)})")
    
    return model
 
 
def remove_importance_masks(model, remove_heads=True, remove_ffn=True):
    """
    Remove registered masks from layers.
    
    Args:
        model: BERT model
        remove_heads: Whether to remove head masks (default: True)
        remove_ffn: Whether to remove FFN masks (default: True)
    """
    for layer in model.bert.encoder.layer:
        if remove_heads:
            layer.head_mask_param = None
        if remove_ffn:
            layer.int_mask_param = None
    
    parts = []
    if remove_heads:
        parts.append("head")
    if remove_ffn:
        parts.append("FFN")
    
    print(f"✓ Removed {' and '.join(parts)} masks")

In [52]:
def apply_head_masking(model, head_importance, num_to_mask):
    """Apply masking to head_mask_param based on importance."""
    
    # Count total heads
    total_heads = sum(len(imp) for imp in head_importance)
    
    print(f"\nMasking {num_to_mask} / {total_heads} heads ({num_to_mask/total_heads*100:.1f}%)")
    
    # Collect all (importance, layer_idx, head_idx)
    all_heads = []
    for layer_idx, layer_imp in enumerate(head_importance):
        for head_idx in range(len(layer_imp)):
            all_heads.append((layer_imp[head_idx].item(), layer_idx, head_idx))
    
    # Sort by importance (ascending - least important first)
    all_heads.sort(key=lambda x: x[0])
    
    # Mask the least important ones
    for i in range(num_to_mask):
        _, layer_idx, head_idx = all_heads[i]
        
        # Set mask to 0 (this head will be masked out)
        with torch.no_grad():
            model.bert.encoder.layer[layer_idx].head_mask_param[head_idx] = 0.0
        
        if i < 5:  # Print first few
            print(f"  Masked layer {layer_idx}, head {head_idx} (importance: {all_heads[i][0]:.6f})")
    
    if num_to_mask > 5:
        print(f"  ... and {num_to_mask - 5} more")
 
 
def apply_neuron_masking(model, neuron_importance, num_to_mask):
    """Apply masking to int_mask_param based on importance."""
    
    # Count total neurons
    total_neurons = sum(len(imp) for imp in neuron_importance)
    
    print(f"\nMasking {num_to_mask} / {total_neurons} neurons ({num_to_mask/total_neurons*100:.1f}%)")
    
    # Collect all (importance, layer_idx, neuron_idx)
    all_neurons = []
    for layer_idx, layer_imp in enumerate(neuron_importance):
        for neuron_idx in range(len(layer_imp)):
            all_neurons.append((layer_imp[neuron_idx].item(), layer_idx, neuron_idx))
    
    # Sort by importance (ascending)
    all_neurons.sort(key=lambda x: x[0])
    
    # Mask the least important ones
    for i in range(num_to_mask):
        _, layer_idx, neuron_idx = all_neurons[i]
        
        # Set mask to 0
        with torch.no_grad():
            model.bert.encoder.layer[layer_idx].int_mask_param[neuron_idx] = 0.0
        
        if i < 5:
            print(f"  Masked layer {layer_idx}, neuron {neuron_idx} (importance: {all_neurons[i][0]:.6f})")
    
    if num_to_mask > 5:
        print(f"  ... and {num_to_mask - 5} more")

In [40]:
def compute_importance_scores(
    model,
    dataloader,
    device,
    max_batches: Optional[int] = None,
    compute_heads: bool = True,
    compute_ffn: bool = True
) -> Tuple[Optional[List[torch.Tensor]], Optional[List[torch.Tensor]]]:
    """
    Compute importance scores for attention heads and/or FFN neurons.
    
    Flexible function that can compute:
    - Only head importance (compute_heads=True, compute_ffn=False)
    - Only FFN importance (compute_heads=False, compute_ffn=True)
    - Both together (compute_heads=True, compute_ffn=True) - single pass!
    
    Args:
        model: BERT model with registered masks
        dataloader: Dataloader for importance computation
        device: torch device
        max_batches: Limit batches (None = use all)
        compute_heads: Whether to compute head importance (default: True)
        compute_ffn: Whether to compute FFN neuron importance (default: True)
    
    Returns:
        (head_importance, neuron_importance):
            - head_importance: List of tensors or None if compute_heads=False
            - neuron_importance: List of tensors or None if compute_ffn=False
    
    Examples:
        # Compute both (single pass - most efficient)
        head_imp, neuron_imp = compute_importance_scores(
            model, dataloader, device,
            compute_heads=True, compute_ffn=True
        )
        
        # Compute only heads
        head_imp, _ = compute_importance_scores(
            model, dataloader, device,
            compute_heads=True, compute_ffn=False
        )
        
        # Compute only FFN
        _, neuron_imp = compute_importance_scores(
            model, dataloader, device,
            compute_heads=False, compute_ffn=True
        )
    """
    if not compute_heads and not compute_ffn:
        raise ValueError("Must compute at least one of: heads or FFN")
    
    model.eval()
    n_layers = len(model.bert.encoder.layer)
    
    # Check which masks are needed and registered
    if compute_heads:
        if model.bert.encoder.layer[0].head_mask_param is None:
            raise ValueError(
                "Head masks not registered! Call register_importance_masks(model, device) first."
            )
    
    if compute_ffn:
        if model.bert.encoder.layer[0].int_mask_param is None:
            raise ValueError(
                "FFN masks not registered! Call register_importance_masks(model, device) first."
            )
    
    # Initialize importance accumulators
    head_importance = None
    neuron_importance = None
    
    if compute_heads:
        head_importance = []
        for layer in model.bert.encoder.layer:
            head_importance.append(torch.zeros_like(layer.head_mask_param))
    
    if compute_ffn:
        neuron_importance = []
        for layer in model.bert.encoder.layer:
            neuron_importance.append(torch.zeros_like(layer.int_mask_param))
    
    # Accumulate gradients
    tot_tokens = 0.0
    n_batches = len(dataloader) if max_batches is None else min(len(dataloader), max_batches)
    
    # Progress bar message
    if compute_heads and compute_ffn:
        desc = "Computing importance (heads + FFN)"
    elif compute_heads:
        desc = "Computing importance (heads only)"
    else:
        desc = "Computing importance (FFN only)"
    
    for batch_idx, batch in enumerate(tqdm(dataloader, total=n_batches, desc=desc)):
        if max_batches and batch_idx >= max_batches:
            break
        
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Forward pass - your forward() will use the registered masks!
        outputs = model(**batch)
        loss = outputs.loss
        
        # Backward pass
        loss.backward()
        
        # Accumulate importance from gradients
        for layer_idx, layer in enumerate(model.bert.encoder.layer):
            if compute_heads and layer.head_mask_param.grad is not None:
                head_importance[layer_idx] += layer.head_mask_param.grad.abs().detach()
            
            if compute_ffn and layer.int_mask_param.grad is not None:
                neuron_importance[layer_idx] += layer.int_mask_param.grad.abs().detach()
        
        # Zero gradients
        model.zero_grad()
        
        tot_tokens += batch['attention_mask'].float().sum().item()
    
    # Normalize
    if compute_heads:
        for layer_idx in range(n_layers):
            head_importance[layer_idx] /= tot_tokens
    
    if compute_ffn:
        for layer_idx in range(n_layers):
            neuron_importance[layer_idx] /= tot_tokens
    
    return head_importance, neuron_importance

In [ ]:
def calculate_proportional_amount(
    model,
    head_amount: float,
    mode: str = 'auto'
) -> Tuple[float, float]:
    """
    Calculate proportional neuron masking amount given head masking amount.
    
    Args:
        model: BERT model
        head_amount: Fraction of heads to mask (e.g., 0.1 for 10%)
        mode: 'auto' | 'equal_ratio' | 'param_ratio'
    
    Returns:
        (head_amount, neuron_amount): Both as fractions
    
    Example:
        # Mask 10% heads, auto-calculate neuron amount
        head_amt, neuron_amt = calculate_proportional_amount(model, 0.1)
        # Returns: (0.1, 0.065) - mask 6.5% of neurons
    """
    # Count total units
    total_heads = 0
    total_neurons = 0
    
    for layer in model.bert.encoder.layer:
        total_heads += layer.attention.self.num_attention_heads
        total_neurons += layer.intermediate.dense.out_features
    
    if mode == 'equal_ratio':
        # Simple: same fraction
        return head_amount, head_amount
    
    elif mode == 'auto' or mode == 'param_ratio':
        # Match parameter reduction
        hidden_dim = model.bert.config.hidden_size
        head_size = model.bert.encoder.layer[0].attention.self.attention_head_size
        
        # Params per head (Q, K, V, output projection)
        params_per_head = 4 * hidden_dim * head_size
        
        # Params per neuron (intermediate + output)
        params_per_neuron = 2 * hidden_dim
        
        # Calculate heads to mask
        heads_to_mask = int(total_heads * head_amount)
        params_removed_heads = heads_to_mask * params_per_head
        
        # Match parameter removal in neurons
        neurons_to_mask = params_removed_heads / params_per_neuron
        neuron_amount = neurons_to_mask / total_neurons
        
        # Clamp to valid range
        neuron_amount = max(0.0, min(1.0, neuron_amount))
        
        return head_amount, neuron_amount
    
    else:
        raise ValueError(f"Unknown mode: {mode}")

In [40]:
def structured_prune_model(model, dataloader, device,
                          head_prune_ratio=0.0, neuron_prune_ratio=0.0,
                          max_batches=None):
    print("="*70)
    print("STRUCTURED PRUNING")
    print("="*70)
    
    if head_prune_ratio == 0 and neuron_prune_ratio == 0:
        print("No pruning requested")
        return
    
    # Step 1: Register masks
    print(f"\n[1/4] Registering importance masks...")
    register_importance_masks(model, device)

In [41]:
structured_prune_model(model, train_dataloader, device, head_prune_ratio=1, max_batches=None)

STRUCTURED PRUNING

[1/4] Registering importance masks...
  Layer 0: Registered masks (12 heads, 3072 neurons)
  Layer 1: Registered masks (12 heads, 3072 neurons)
  Layer 2: Registered masks (12 heads, 3072 neurons)
  Layer 3: Registered masks (12 heads, 3072 neurons)
  Layer 4: Registered masks (12 heads, 3072 neurons)
  Layer 5: Registered masks (12 heads, 3072 neurons)
  Layer 6: Registered masks (12 heads, 3072 neurons)
  Layer 7: Registered masks (12 heads, 3072 neurons)
  Layer 8: Registered masks (12 heads, 3072 neurons)
  Layer 9: Registered masks (12 heads, 3072 neurons)
  Layer 10: Registered masks (12 heads, 3072 neurons)
  Layer 11: Registered masks (12 heads, 3072 neurons)


In [1]:
3072 / 12

256.0

In [47]:
head_importance, neuron_importance = compute_importance_scores(model, val_dataloader, device, max_batches=50)

Computing importance (heads + FFN): 100%|██████████| 13/13 [00:01<00:00, 10.49it/s]


In [42]:
model.bert.encoder.layer[0].head_mask_param, model.bert.encoder.layer[0].int_mask_param

(None, None)

In [43]:
register_importance_masks(model, device)

  Layer 0: Registered masks (12 heads, 3072 neurons)
  Layer 1: Registered masks (12 heads, 3072 neurons)
  Layer 2: Registered masks (12 heads, 3072 neurons)
  Layer 3: Registered masks (12 heads, 3072 neurons)
  Layer 4: Registered masks (12 heads, 3072 neurons)
  Layer 5: Registered masks (12 heads, 3072 neurons)
  Layer 6: Registered masks (12 heads, 3072 neurons)
  Layer 7: Registered masks (12 heads, 3072 neurons)
  Layer 8: Registered masks (12 heads, 3072 neurons)
  Layer 9: Registered masks (12 heads, 3072 neurons)
  Layer 10: Registered masks (12 heads, 3072 neurons)
  Layer 11: Registered masks (12 heads, 3072 neurons)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [44]:
model.bert.encoder.layer[0].head_mask_param.shape, model.bert.encoder.layer[0].int_mask_param.shape

(torch.Size([12]), torch.Size([3072]))

In [52]:
remove_importance_masks(model)

✓ Removed head and FFN masks


In [53]:
model.bert.encoder.layer[0].head_mask_param, model.bert.encoder.layer[0].int_mask_param

(None, None)

In [48]:
head_importance, neuron_importance = compute_importance_scores(model, val_dataloader, device, max_batches=50)

Computing importance (heads + FFN): 100%|██████████| 13/13 [00:01<00:00, 10.54it/s]


In [49]:
apply_head_masking(model, head_importance, num_to_mask=6)


Masking 6 / 144 heads (4.2%)
  Masked layer 11, head 3 (importance: 0.000000)
  Masked layer 11, head 1 (importance: 0.000000)
  Masked layer 11, head 0 (importance: 0.000001)
  Masked layer 11, head 10 (importance: 0.000001)
  Masked layer 11, head 6 (importance: 0.000001)
  ... and 1 more


In [50]:
model.bert.encoder.layer[11].head_mask_param

Parameter containing:
tensor([0., 0., 1., 0., 1., 1., 0., 1., 1., 1., 0., 1.], device='cuda:1',
       requires_grad=True)

In [51]:
model.bert.encoder.layer[10].head_mask_param

Parameter containing:
tensor([1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1.], device='cuda:1',
       requires_grad=True)

In [53]:
apply_neuron_masking(model, neuron_importance, 256)


Masking 256 / 36864 neurons (0.7%)
  Masked layer 0, neuron 2165 (importance: 0.000000)
  Masked layer 11, neuron 2120 (importance: 0.000000)
  Masked layer 11, neuron 861 (importance: 0.000000)
  Masked layer 0, neuron 2172 (importance: 0.000000)
  Masked layer 11, neuron 605 (importance: 0.000000)
  ... and 251 more


In [56]:
sum(model.bert.encoder.layer[11].int_mask_param), len(model.bert.encoder.layer[11].int_mask_param)

(tensor(2886., device='cuda:1', grad_fn=<AddBackward0>), 3072)

In [46]:
total_heads = sum(len(imp) for imp in head_importance)
total_heads

144

In [47]:
num_to_mask = 6

In [48]:
all_heads = []
for layer_idx, layer_imp in enumerate(head_importance):
    for head_idx in range(len(layer_imp)):
        all_heads.append((layer_imp[head_idx].item(), layer_idx, head_idx))

In [49]:
all_heads

[(2.5404695406905375e-05, 0, 0),
 (2.332831354578957e-05, 0, 1),
 (8.99463066161843e-06, 0, 2),
 (1.4079963875701651e-05, 0, 3),
 (9.609687367628794e-06, 0, 4),
 (4.947947309119627e-05, 0, 5),
 (1.4488286069536116e-05, 0, 6),
 (1.4784608538320754e-05, 0, 7),
 (9.092649270314723e-05, 0, 8),
 (3.4979584597749636e-05, 0, 9),
 (1.3139299881004263e-05, 0, 10),
 (1.6921896531130187e-05, 0, 11),
 (1.509594585513696e-05, 1, 0),
 (2.9622067813761532e-05, 1, 1),
 (2.4219580154749565e-05, 1, 2),
 (8.717278433323372e-06, 1, 3),
 (2.2719366825185716e-05, 1, 4),
 (8.394787073484622e-06, 1, 5),
 (3.691059828270227e-05, 1, 6),
 (6.170330016175285e-05, 1, 7),
 (5.2440096624195576e-05, 1, 8),
 (1.0566367564024404e-05, 1, 9),
 (1.0887352800637018e-05, 1, 10),
 (7.379221642622724e-05, 1, 11),
 (2.061215491266921e-05, 2, 0),
 (2.068457797577139e-05, 2, 1),
 (1.564724516356364e-05, 2, 2),
 (3.376562017365359e-05, 2, 3),
 (8.28805877972627e-06, 2, 4),
 (1.5256959159160033e-05, 2, 5),
 (0.00011151980288559571

In [51]:
all_heads.sort(key=lambda x: x[0])
all_heads

[(2.7947660896643356e-07, 11, 3),
 (3.4678714655456133e-07, 11, 1),
 (7.295998898371181e-07, 11, 0),
 (9.005166816677956e-07, 11, 10),
 (1.3293049505591625e-06, 11, 6),
 (1.3923838650953257e-06, 10, 4),
 (1.5024104413896566e-06, 10, 3),
 (1.511905566076166e-06, 11, 11),
 (1.949402530954103e-06, 9, 9),
 (2.141539880540222e-06, 10, 2),
 (2.4612495508335996e-06, 10, 6),
 (2.4674695850990247e-06, 11, 5),
 (2.5700992409838364e-06, 11, 4),
 (2.7923806555918418e-06, 9, 1),
 (2.949204827018548e-06, 8, 5),
 (3.049012775591109e-06, 9, 3),
 (3.055782826777431e-06, 7, 6),
 (3.0921123652660754e-06, 8, 4),
 (3.1005404252937296e-06, 9, 6),
 (3.1175886761047877e-06, 10, 9),
 (3.346185394548229e-06, 5, 1),
 (3.4334541396674467e-06, 11, 7),
 (3.4448153201083187e-06, 7, 3),
 (3.514635409374023e-06, 10, 11),
 (3.920463313988876e-06, 10, 10),
 (4.222330517222872e-06, 10, 7),
 (4.429245109349722e-06, 8, 6),
 (4.612175871443469e-06, 7, 7),
 (4.673008788813604e-06, 6, 3),
 (4.8565984798187856e-06, 9, 4),
 (4.

In [52]:
for i in range(num_to_mask):
    _, layer_idx, head_idx = all_heads[i]
    
    # Set mask to 0 (this head will be masked out)
    with torch.no_grad():
        model.bert.encoder.layer[layer_idx].head_mask_param[head_idx] = 0.0
    
    if i < 5:  # Print first few
        print(f"  Masked layer {layer_idx}, head {head_idx} (importance: {all_heads[i][0]:.6f})")

  Masked layer 11, head 3 (importance: 0.000000)
  Masked layer 11, head 1 (importance: 0.000000)
  Masked layer 11, head 0 (importance: 0.000001)
  Masked layer 11, head 10 (importance: 0.000001)
  Masked layer 11, head 6 (importance: 0.000001)


In [54]:
model.bert.encoder.layer[11].head_mask_param

Parameter containing:
tensor([0., 0., 1., 0., 1., 1., 0., 1., 1., 1., 0., 1.], device='cuda:1',
       requires_grad=True)

In [55]:
model.bert.encoder.layer[10].head_mask_param

Parameter containing:
tensor([1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1.], device='cuda:1',
       requires_grad=True)